<a href="https://colab.research.google.com/github/MelisaYasak/transformer_arch/blob/main/sherlockGPT_biagram_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content/drive/MyDrive/transformers

/content/drive/MyDrive/transformers


# Bigram Modeli

Bigram bir sonraki token'ı tahmin etmek içi sadece o stepdeki token'ı dikkate alan basic bir dil modeli.

**_P(x_t+1 | x_t)_**

Dipnot: Bağlamı sadece bir karakterdir.

## Data Prep

In [2]:
from pathlib import Path

In [3]:
text = Path("sherlock.txt").read_text(
    encoding="utf-8-sig"
)

In [4]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

In [5]:
start = text.find(start_marker)
end = text.find(end_marker)

In [6]:
if start != -1:
    text = text[text.find("\n", start) + 1:]

if end != -1:
    text = text[:text.find(end_marker)]

In [7]:
text = text.strip()

In [8]:
print("Karakter sayısı:", len(text))
print(text[:300])

Karakter sayısı: 562202
The Adventures of Sherlock Holmes

by Arthur Conan Doyle


Contents

   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of the 


# Tokenizer ve Dataset

### Dataset oluştur

In [9]:
import torch

In [10]:
torch.manual_seed(42)

In [11]:
chars = sorted(set(text))

In [12]:
vocab_size = len(chars)
vocab_size

88

In [13]:
stoi = {ch: i for i, ch in enumerate(chars)}

itos = {i: ch for ch, i in stoi.items()}

In [14]:
encode = lambda s: [stoi[c] for c in s]

decode = lambda ids: "".join(itos[i] for i in ids)

In [15]:
data = torch.tensor(encode(text), dtype=torch.long)

In [16]:
# eğitimden önce veriyi train ve test olarak ayıracağız

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [17]:
print("Vocabulary size:", vocab_size)
print("Train shape:", train_data.shape)
print("Validation shape:", val_data.shape)
print("İlk karakterler:", decode(data[:50].tolist()))

Vocabulary size: 88
Train shape: torch.Size([505981])
Validation shape: torch.Size([56221])
İlk karakterler: The Adventures of Sherlock Holmes

by Arthur Conan


### Modeli Kur

In [18]:
import torch.nn as nn
import torch.nn.functional as F

In [19]:
class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()

    # Bir embedding oluşturacağız.
    # Tabloda vocab_size kadar farklı token bulunacak
    # ve her token vocab_size kadar boyutlu vektöre dönüştürülecek.
    # Bigram modelinde her karakter için vocabulary boyutunda bir logit vektörü kullanıyoruz.
    self.token_embedding_table = nn.Embedding(
        vocab_size,
        vocab_size
    )

  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx)

    loss = None

    if targets is not None:
      B, T, C = logits.shape

      logits_flat = logits.reshape(B * T, C)
      targets_flat = targets.reshape(B * T)

      loss = F.cross_entropy(logits_flat, targets_flat)

    return logits, loss

In [20]:
batch_size = 16
block_size = 8

In [21]:
def get_batch(split):
  source = train_data if split == "train" else val_data

  ix = torch.randint(
      0,
      len(source) - block_size,
      (batch_size,)
  )

  x = torch.stack([
      source[i:i + block_size]
      for i in ix
  ])

  y = torch.stack([
      source[i+1:i + block_size + 1]
      for i in ix
  ])

  return x, y

xb, yb = get_batch("train")

In [22]:
print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("Input:", decode(xb[0].tolist()))
print("Target:", decode(yb[0].tolist()))

Input shape: torch.Size([16, 8])
Target shape: torch.Size([16, 8])
Input: : It was
Target:  It was.


### Eğitime Başla

In [23]:
model = BigramLanguageModel(vocab_size)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001
)

xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("Başlangıç loss:", loss.item())

Başlangıç loss: 4.953321933746338


In [24]:
for step in range(10000):
  xb, yb = get_batch("train")
  logits, loss = model(xb, yb)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if step % 200 == 0:
    print(
            f"Step {step} | "
            f"Train batch loss: {loss.item():.4f}"
        )

Step 0 | Train batch loss: 5.1560
Step 200 | Train batch loss: 4.8816
Step 400 | Train batch loss: 4.6854
Step 600 | Train batch loss: 4.4502
Step 800 | Train batch loss: 4.2775
Step 1000 | Train batch loss: 4.0147
Step 1200 | Train batch loss: 3.9220
Step 1400 | Train batch loss: 3.7821
Step 1600 | Train batch loss: 3.5867
Step 1800 | Train batch loss: 3.6534
Step 2000 | Train batch loss: 3.5122
Step 2200 | Train batch loss: 3.4434
Step 2400 | Train batch loss: 3.2040
Step 2600 | Train batch loss: 3.1138
Step 2800 | Train batch loss: 2.9315
Step 3000 | Train batch loss: 2.9604
Step 3200 | Train batch loss: 2.8752
Step 3400 | Train batch loss: 2.8289
Step 3600 | Train batch loss: 2.7359
Step 3800 | Train batch loss: 2.8033
Step 4000 | Train batch loss: 2.6493
Step 4200 | Train batch loss: 2.7814
Step 4400 | Train batch loss: 2.6174
Step 4600 | Train batch loss: 2.5379
Step 4800 | Train batch loss: 2.7287
Step 5000 | Train batch loss: 2.8010
Step 5200 | Train batch loss: 2.5523
Step 540

In [25]:
model.eval()

with torch.no_grad():
  xb, yb = get_batch("val")

  logits, val_loss = model(xb, yb)

print("Validation batch loss:", val_loss.item())


Validation batch loss: 2.3787941932678223


### Let's Go to the Test

In [26]:
@torch.no_grad()
def generate(model, start_id, max_new_tokens=200):
  model.eval()

  idx = torch.tensor([[start_id]], dtype=torch.long)

  for _ in range(max_new_tokens):

    logits, _ = model(idx)

    last_logits = logits[:, -1, :]

    probs = F.softmax(last_logits, dim=-1)

    next_id = torch.multinomial(
        probs,
        num_samples=1
    )

    idx = torch.cat([idx, next_id], dim=1)

  return decode(idx[0].tolist())

print(generate(model, stoi["S"]))

Sigengo sthithen sthen hin r brkn thed ins STHer
tuncopathended ousir, u


“The j_jTE3zve in AVomirlomporinig aneen MResoron omangn onar wangou
thade t ig fow athiengh! Twe d s. fu
“Thnd bofrvo w ild h
